# Análisis y Preparación de Datos: Ejemplo Aplicado de Manchas Solares

Este notebook es un ejemplo aplicado de **Análisis y Preparación de Datos** en el contexto de la Ingeniería Mecatrónica. Se realiza una limpieza exhaustiva, análisis comparativo, reducción de dimensionalidad (PCA) y detección de outliers sobre un dataset de clasificación McIntosh de manchas solares.

**Etapas clave:**
1. Carga y Limpieza inicial de nulos.
2. Normalización Min-Max [0, 1] de metadatos (parámetros físicos).
3. Análisis de componentes principales (PCA).
4. Detección y retiro de outliers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from mpl_toolkits.mplot3d import Axes3D

# Configuración de estética
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Carga y Limpieza Inicial

In [ ]:
df_raw = pd.read_csv('sunspot_data.csv')

# Limpieza de nulos
df = df_raw.dropna(axis=0, how='any').copy()
df = df.dropna(axis=1, how='all')

# Ingeniería de características
df['mc_Z'] = df['mcintosh_full'].str[0]
df['mc_P'] = df['mcintosh_full'].str[1]
df['mc_C'] = df['mcintosh_full'].str[2]

print(f"Dataset cargado y limpio de nulos. Registros: {df.shape[0]}")

## 2. Visualización: Distribuciones Iniciales (Antes de Limpieza Final)

En esta sección visualizamos la frecuencia de las clases McIntosh de forma independiente y destacada.

In [ ]:
# 2.1 Distribución de McIntosh Full (Combinado)
plt.figure(figsize=(15, 6))
sns.countplot(data=df, x='mcintosh_full', hue='mcintosh_full', legend=False, 
              order=df['mcintosh_full'].value_counts().index, palette='viridis')
plt.title('Distribución Inicial: McIntosh Full (Clasificación Completa)', fontsize=15)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 2.2 Distribución de Caracteres Individuales
fig, axes = plt.subplots(3, 1, figsize=(12, 18))
chars = [('mc_Z', 'Z (Clase de Mancha)'), ('mc_P', 'P (Tipo de Penumbra)'), ('mc_C', 'C (Distribución)')]

for i, (col, label) in enumerate(chars):
    sns.countplot(data=df, x=col, hue=col, legend=False, ax=axes[i], 
                  order=df[col].value_counts().index, palette='magma')
    axes[i].set_title(f'Distribución Inicial: Carácter {label}', fontsize=14)
    axes[i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## 3. Normalización Min-Max [0, 1] de Metadatos

Escalamos los parámetros físicos al rango [0, 1] para uniformizar las entradas.

In [ ]:
features = ['area', 'dist_angular', 'z_length_deg', 'num_spots_det', 'area_total', 
            'fill_ratio', 'elongation', 'pen_diam_ns_deg', 'pen_symmetry', 'pen_irregularity']

X = df[features]

# Normalización Min-Max
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Visualización de datos normalizados
df_norm = pd.DataFrame(X_scaled, columns=features)
print("Metadatos normalizados en rango [0, 1]:")
display(df_norm.head())
print(f"Mínimo global: {df_norm.min().min()}, Máximo global: {df_norm.max().max()}")

## 4. Estadísticos Agrupados Estilizados

In [ ]:
stats_summary = df.groupby('mcintosh_full')[features].mean()
print("Medias por Clase (Datos Originales):")
try:
    display(stats_summary.style.background_gradient(cmap='YlGnBu').format("{:.2f}"))
except Exception:
    display(stats_summary)

## 5. PCA y Detección de Outliers

In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

iso = IsolationForest(contamination=0.05, random_state=42)
df['outlier'] = iso.fit_predict(X_scaled)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_pca[df['outlier'] == 1, 0], X_pca[df['outlier'] == 1, 1], X_pca[df['outlier'] == 1, 2], 
           c='blue', alpha=0.3, label='Normal')
ax.scatter(X_pca[df['outlier'] == -1, 0], X_pca[df['outlier'] == -1, 1], X_pca[df['outlier'] == -1, 2], 
           c='red', marker='x', s=50, label='Outlier')
ax.set_title('Visualización PCA 3D y Detección de Anomalías')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_zlabel('PC3')
ax.legend()
plt.show()

## 6. Resultados Finales: Dataset Limpio y Sin Outliers

In [ ]:
df_clean = df[df['outlier'] == 1].copy()
print(f"Limpieza completada. Dataset final: {df_clean.shape[0]} registros.")

# Histograma Final Destacado
plt.figure(figsize=(15, 6))
sns.countplot(data=df_clean, x='mcintosh_full', hue='mcintosh_full', legend=False, 
              order=df_clean['mcintosh_full'].value_counts().index, palette='viridis')
plt.title('Distribución Final: McIntosh Full (Sin Outliers)', fontsize=15)
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_clean.to_csv('sunspot_data_clean.csv', index=False)
print("Base de datos 'sunspot_data_clean.csv' generada con éxito.")